In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error 
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import KFold
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [2]:
# load experimental data into pandas dataframe

df = pd.read_csv("team-a.csv")
df = df.drop(['formula'],axis=1)

In [ ]:
# split experimental data - use all features for nn 

X = df.drop(['gap expt'],axis=1).values

y = df['gap expt'].values
y = y.reshape(-1,1)

# 80% train, 10% validation, 10% test data split:
# using holdout method, cross val could be better if concerned the dataset is not fairly represented in test set

X_train,X_test_val,y_train,y_test_val = train_test_split(X,y,test_size=0.2,random_state=42)
X_val,X_test,y_val,y_test = train_test_split(X_test_val,y_test_val,test_size=0.5,random_state=42)

In [21]:
# scaling data, converting to tensors and loading dataloader

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1,1))
X_val_scaled = scaler_X.transform(X_val)
y_val_scaled = scaler_y.transform(y_val.reshape(-1,1))
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test.reshape(-1,1))

# convert to tensors
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train_scaled)
X_val_tensor = torch.FloatTensor(X_val_scaled)
y_val_tensor = torch.FloatTensor(y_val_scaled)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test_scaled)

# create dataloader - this feeds data in batches during training 
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor),batch_size=32,shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor,y_val_tensor),batch_size=32, shuffle=True)

In [ ]:
# define nn class:
# Suggestion: Extend so that both depth and width of NN can be set through the constructor arguments

class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.relu(out)
        out = self.fc4(out)

        return out

In [ ]:
# implement early stopping
# early stopping can slightly overfit to validation set as it is optimised for val, not test

class EarlyStopping:
    def __init__(self, patience, delta):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def load_best_model(self, model):
        model.load_state_dict(self.best_model_state)

In [ ]:
# represents uncertainty in data splitting

from sklearn.utils import resample

def bootstrap_mae_ci(y_true, y_pred, n_bootstrap=1000, confidence=0.95):
    """Calculate bootstrap confidence interval for MAE"""
    maes = []
    n_samples = len(y_true)
    
    for i in range(n_bootstrap):
        # Resample with replacement
        indices = resample(range(n_samples), n_samples=n_samples)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]
        
        # Calculate MAE for this bootstrap sample
        mae_boot = np.mean(np.abs(y_true_boot - y_pred_boot))
        maes.append(mae_boot)
    
    # Calculate confidence interval
    alpha = 1 - confidence
    lower = np.percentile(maes, alpha/2 * 100)
    upper = np.percentile(maes, (1 - alpha/2) * 100)
    
    return lower, upper

In [25]:
input_size = X.shape[1]
hidden_size = 321
num_classes = 1

model = SimpleNet(input_size, hidden_size, num_classes)
early_stopping = EarlyStopping(patience=50, delta=0.01)

criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0014499134483449525)

# Number of complete passes through the dataset
num_epochs = 200

# keep track of the loss for each epoch
train_losses = []
val_losses = []

# Start the training loop
for epoch in range(num_epochs):
    # Set the model to training mode
    model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
       
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Iterate over the validation data and compute the loss
    model.eval()
    val_loss = 0.0

    # turn off gradients 
    with torch.no_grad():
        for batch_X, batch_y in val_loader:

            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()

    train_losses.append(train_loss/len(train_loader))
    val_losses.append(val_loss/len(val_loader))
    print(f"Epoch [{epoch+1}/{num_epochs}], Train loss: {train_losses[-1]:.4f}, Val loss: {val_losses[-1]:.4f}")

    # early stopping and final evaluation from test set:
    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print("Early stopping")
        with torch.no_grad():
            y_pred_scaled = model(X_test_tensor)

            y_pred = scaler_y.inverse_transform(y_pred_scaled.numpy())
            y_test = scaler_y.inverse_transform(y_test_tensor.numpy())
            test_mae = np.mean(np.abs(y_pred - y_test))

            # bootstrap confidence interval:
    
            lower, upper = bootstrap_mae_ci(y_test, y_pred)
    
            print(f"MAE: {test_mae:.4f} ± {(upper - lower)/2:.4f}")
            
        break

    early_stopping.load_best_model(model)

Epoch [1/200], Train loss: 0.4810, Val loss: 8.1769
Epoch [2/200], Train loss: 0.3943, Val loss: 88.8348
Epoch [3/200], Train loss: 0.3696, Val loss: 27.4898
Epoch [4/200], Train loss: 0.3579, Val loss: 44.2099
Epoch [5/200], Train loss: 0.3341, Val loss: 7.3484
Epoch [6/200], Train loss: 0.3210, Val loss: 27.0055
Epoch [7/200], Train loss: 0.3088, Val loss: 66.9343
Epoch [8/200], Train loss: 0.3003, Val loss: 15.5278
Epoch [9/200], Train loss: 0.2872, Val loss: 42.7915
Epoch [10/200], Train loss: 0.2785, Val loss: 22.6914
Epoch [11/200], Train loss: 0.2749, Val loss: 22.4872
Epoch [12/200], Train loss: 0.2591, Val loss: 51.5688
Epoch [13/200], Train loss: 0.2525, Val loss: 49.4478
Epoch [14/200], Train loss: 0.2418, Val loss: 27.4738
Epoch [15/200], Train loss: 0.2479, Val loss: 5.6127
Epoch [16/200], Train loss: 0.2514, Val loss: 79.1607
Epoch [17/200], Train loss: 0.2464, Val loss: 55.6979
Epoch [18/200], Train loss: 0.2320, Val loss: 34.9335
Epoch [19/200], Train loss: 0.2312, Val 